# FNO Training on PFC Data (Notebook 2 of 2)

Trains the Fourier Neural Operator surrogate on the dataset created by
Notebook 1 (`pfc_dataset_108.zip` in your Drive), then evaluates it.

**Runtime: GPU** — `Runtime → Change runtime type → T4 GPU` **before** running.

Run cells in order. Training uses a pure MSE loss; every 10 epochs it also
prints a free-running *rollout* MSE + mass drift on validation trajectories —
that rollout number is the real measure of surrogate quality.


### 1. Mount Drive, get the code, install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy pyyaml matplotlib tensorboard
!nvidia-smi -L   # should print a GPU; if not, fix the runtime type first

### 1b. Hard stop if there is no GPU
Training on Colab's CPU takes 10-30x longer (this is the usual cause of
"it's been running forever"). This cell refuses to continue without a GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "NO GPU DETECTED. Runtime -> Change runtime type -> T4 GPU, "
    "then Runtime -> Run all again.")
print("GPU OK:", torch.cuda.get_device_name(0))

### 2. Pull the dataset back from Drive

In [ ]:
!unzip -q /content/drive/MyDrive/pfc_dataset_108.zip
!ls data_pfc_large | head -4
!tail -3 data_pfc_large/manifest.csv

### 3. Point the training config at the dataset
`split_by: config` keeps both noise realizations of one `(r, n0, seed_type)`
in the same train/val/test split — no near-duplicate leakage.

In [ ]:
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["data_dir"] = "data_pfc_large"
cfg["data"]["split_by"] = "config"
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
print(cfg["data"])

### 4. Train (pure-MSE PFC baseline; early-stops automatically)

In [ ]:
!python train_fno.py --config config.yaml

### 5. Save the trained model to Drive IMMEDIATELY (Colab wipes local files on disconnect)

In [ ]:
!cp runs/fno_baseline/best.pt /content/drive/MyDrive/fno_pfc_best.pt
print("Checkpoint in Drive: MyDrive/fno_pfc_best.pt")

### 6. Evaluate: one-step MSE, multi-step rollout MSE, mass error, figures
With `enforce_mass: true` (the default) the mass error should be ~1e-7 or better.

In [ ]:
!python evaluate_fno.py --config config.yaml --checkpoint runs/fno_baseline/best.pt

from IPython.display import Image, display
import glob
for png in sorted(glob.glob("runs/fno_baseline/eval/*.png")):
    display(Image(png))

### 6b. Results table: which regime is failing?
Loads the per-trajectory CSV the evaluator wrote and groups rollout MSE by
`seed_type` and by `r`. This is the diagnostic for deciding what to change
next (e.g. rebalance seeds vs. focus on a quench regime).

In [ ]:
import pandas as pd
df = pd.read_csv('runs/fno_baseline/eval/per_trajectory.csv')
print('Per-trajectory (worst first):')
display(df.sort_values('rollout_mse', ascending=False))
print('\nMean rollout MSE by seed_type:')
display(df.groupby('seed_type')['rollout_mse'].agg(['mean','count']))
print('\nMean rollout MSE by r:')
display(df.groupby('r')['rollout_mse'].agg(['mean','count']))

### 7. (Optional) keep the evaluation figures in Drive too

In [ ]:
!zip -r -q fno_eval_figures.zip runs/fno_baseline/eval
!cp fno_eval_figures.zip /content/drive/MyDrive/
print("Figures in Drive: MyDrive/fno_eval_figures.zip")